# Post a Site-Directed Mutagenesis Design to Teselagen

So that we can use Teselagen to design primers. These builds are two-part gibsons, with one split in the antibiotic marker and the other split at the site of the mutation.

Read through the required inputs to populate this script, and then run through. Once finished, you can follow the [FolDE SDM Protocol](https://jbei.teselagen.com/client/entries/a346c918-991a-4768-bf79-42cad5662a61) in the FolDE project in Teselagen to finish turning the build into an assembly report, ordering DNA, and using the robots to finish the build.

Required inputs:
* **CAMPAIGN_ID:** Name of this campaign. Will be used in design ID and the input genbank file names, and output teselagen build names.
* **DESIGN_ID:** What you want this called in Teselagen
* **TEMPLATE_PLASMID_REPO:** A collection of genbank files for plasmids that you have in stock, that you want to use when building this next round of mutants.
    * each genbank file should be indexed at zero within the antibiotic marker - that defines the first boundary for the gibson.
    * there should be EXACTLY one CDS feature in the genbank, and it should be on the forward strand - this is the CDS that we will mutagenize. It should include the start and stop codons.
    * the genbanks should be named like <campaign_id>_<seq_id>.gb where seq_id is either WT for the wild type sequence or follows the format described above.
* **OUTPUT_CSV_FPATH:** Where to store the mapping from teselagen design name to the target seq_id. *Keep that around, it's the rosetta stone for mapping your plasmids.*
* **TESELAGEN_OTP:** A one-time-password for uploading data to teselagen. These are specific to each user and should not ever be shared. Eg, if Justin is uploading a design to Teselagen, he should get his own OTP, and remove it from the file before saving. You can get one from `Settings-> API Password` within the application.
* **TESELAGEN_PROJECT_ID:** Which project you want the design (and templates) pasted into. You can get the project ID by going to `Settings -> Projects` and modifying the table to display the `ID` column, then right-clicking and copying that ID cell value into the notebook. It should look like 631fdf1d-5b6d-4e03-8f0c-a0f23f5066ed, which is the FolDE project ID.
* **TESELAGEN_USERNAME:** Your username.
* **NEW_SEQ_IDS:** a list of seq_ids that you want to create (eg, D104G_G429R is a D->G mutation at 104 and G->R mutation at 429; allele_ids should always be sorted by locus: 104 comes before 429). This could come, for example, from FolDE in Foldy.

In [ ]:
CAMPAIGN_ID = 'TY_Pop2'

DESIGN_ID = f'TY_Pop2_R2_test'

TEMPLATE_PLASMID_REPO = 'notebooks/jacob/round1/new_plasmids'

OUTPUT_CSV_FPATH = 'notebooks/jacob/round2/TY_Pop2_R2_test.csv'

TESELAGEN_OTP = ''
TESELAGEN_PROJECT_ID = ''  # FolDE project: 631fdf1d-5b6d-4e03-8f0c-a0f23f5066ed
TESELAGEN_USERNAME = ''  # Replace this with your username

assert TESELAGEN_OTP, 'TESELAGEN_OTP must be set'
assert TESELAGEN_PROJECT_ID, 'TESELAGEN_PROJECT_ID must be set'

NEW_SEQ_IDS = '''Q70K
D186N
K234E
E310R
E360D
F420L
R200K
I182V
D104G_G429R
D104R_G429R
D152E_G429R
F265R_G429R
F61L_F265R
F61L_G429R
G416R_G429R
G429R_T458R
G429R_T473A
H134R_G429R
H8K_G429R
H8Q_G429R
H8R_G429R
K234E_G429R
M320R_G429R
M381Y_G429R
Q184S_G429R
Q237E_G429R
S125P_G429R
S371R_G429R
T9R_G429R
W412L_G429R
Y128M_G429R
Y327A_G429R
Y405A_G429R
Y327P_G429R'''.split('\n')

In [24]:
# Utility function for finding the 
import logging
from collections import defaultdict
import glob
from app.helpers.sequence_util import allele_set_to_seq_id
from pathlib import Path
from app.helpers.sequence_util import sort_seq_id_list_no_verification
import pandas as pd


def get_seq_id_to_build_tuples(seq_ids, campaign_id, previous_round_plasmid_repo):
    base_seq_ids = []
    for fpath in Path(previous_round_plasmid_repo).glob(f'{campaign_id}*.gb'):
        base_seq_ids.append(fpath.stem.split('-')[1])

    possible_base_frequency = defaultdict(int)
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        for possible_novel_allele in new_seq_id_allele_set:
            required_base_seq_id = allele_set_to_seq_id(new_seq_id_allele_set - {possible_novel_allele})
            if required_base_seq_id in base_seq_ids:
                possible_base_frequency[required_base_seq_id] += 1
    possible_base_frequency = dict(possible_base_frequency)

    seq_id_to_build_tuple = {}
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        if len(new_seq_id_allele_set) == 1:
            seq_id_to_build_tuple[new_seq_id] = ('WT', list(new_seq_id_allele_set)[0])
            continue

        # Always prioritize more common possible bases.
        for possible_base, frequency in sorted(possible_base_frequency.items(), key=lambda x: x[1], reverse=True):
            possible_base_allele_set = set(possible_base.split('_'))
            if possible_base_allele_set.issubset(new_seq_id_allele_set):
                required_new_allele_set = new_seq_id_allele_set - possible_base_allele_set
                if len(required_new_allele_set) == 1:
                    seq_id_to_build_tuple[new_seq_id] = (possible_base, list(required_new_allele_set)[0])
                    break
        else:
            logging.error(f"No base sequence found for {new_seq_id}. Skipping.")

    seq_id_to_build_tuple = dict(seq_id_to_build_tuple)
    return seq_id_to_build_tuple


In [ ]:
seq_id_to_build_tuples = get_seq_id_to_build_tuples(NEW_SEQ_IDS, CAMPAIGN_ID, TEMPLATE_PLASMID_REPO)

round2_seq_dict_list = []
for seq_index, new_seq_id in enumerate(sort_seq_id_list_no_verification(NEW_SEQ_IDS)):
    base_seq_id, new_allele_id = seq_id_to_build_tuples[new_seq_id]
    round2_seq_dict_list.append({
        'campaign_id': CAMPAIGN_ID,
        'design_id': DESIGN_ID,
        'seq_id': new_seq_id,
        'teselagen_plasmid_id': f'{DESIGN_ID}_{seq_index+1:04}',
        'base_seq_id': base_seq_id,
        'new_allele_id': new_allele_id,
    })

round2_seq_df = pd.DataFrame(round2_seq_dict_list)

round2_seq_df.to_csv(OUTPUT_CSV_FPATH, index=False)
round2_seq_df


In [22]:
# Utility class for building Teselagen design
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import re
import copy
import hashlib


############################################################
# Teselagen design data classes
############################################################

class TeselagenDesignPart:
    """A part in a Teselagen construct.

    Parameters
    ----------
    nucleic_acid_seq : str, optional
        A short nucleic‑acid sequence (to be synthesised).
    gb_tuple : tuple(str, int, int), optional
        (genbank_path, start, stop) referencing a slice in an existing plasmid.
    """
    def __init__(self, nucleic_acid_seq=None, gb_tuple=None):
        if (nucleic_acid_seq is None) == (gb_tuple is None):
            raise ValueError("Provide *either* nucleic_acid_seq or gb_tuple, not both.")
        self.nucleic_acid_seq = nucleic_acid_seq
        self.gb_tuple = gb_tuple  # (path,start,stop)

    # convenient fingerprint (hashable key) ----------------------------------
    @property
    def _key(self):
        if self.nucleic_acid_seq is not None:
            return ("seq", self.nucleic_acid_seq)
        path, start, stop = self.gb_tuple
        return ("gb", Path(path).resolve().as_posix(), start, stop)

    def to_dict(self):
        if self.nucleic_acid_seq:
            return {"type": "peptide", "sequence": self.nucleic_acid_seq}
        path, start, stop = self.gb_tuple
        return {"type": "reference", "path": path, "start": start, "stop": stop}

    def __repr__(self):
        if self.nucleic_acid_seq:
            return f"TeselagenDesignPart(nucleic_acid_seq={self.nucleic_acid_seq[:10]}… )"
        return f"TeselagenDesignPart(gb_tuple={self.gb_tuple})"


class TeselagenDesignConstruct:
    """Container for parts belonging to one mutant design."""

    def __init__(self, name):
        self.name = name
        self.parts = []  # list[ TeselagenDesignPart ]

    def add_part(self, part: "TeselagenDesignPart"):
        self.parts.append(part)

    def to_dict(self):
        return {"name": self.name, "parts": [p.to_dict() for p in self.parts]}

    def __repr__(self):
        return f"TeselagenDesignConstruct(name={self.name}, parts={self.parts})"


############################################################
# TeselagenDesignBuilder – builds mutants and can emit JSON
############################################################

class TeselagenDesignBuilder:
    """Accumulates Teselagen constructs and can mutate targets, then export."""

    # Simplified codon table (all available codons per AA)
    CODON_TABLE = {
        "A": ["GCT", "GCC", "GCA", "GCG"],
        "R": ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"],
        "N": ["AAT", "AAC"],
        "D": ["GAT", "GAC"],
        "C": ["TGT", "TGC"],
        "Q": ["CAA", "CAG"],
        "E": ["GAA", "GAG"],
        "G": ["GGT", "GGC", "GGA", "GGG"],
        "H": ["CAT", "CAC"],
        "I": ["ATT", "ATC", "ATA"],
        "L": ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"],
        "K": ["AAA", "AAG"],
        "M": ["ATG"],
        "F": ["TTT", "TTC"],
        "P": ["CCT", "CCC", "CCA", "CCG"],
        "S": ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"],
        "T": ["ACT", "ACC", "ACA", "ACG"],
        "W": ["TGG"],
        "Y": ["TAT", "TAC"],
        "V": ["GTT", "GTC", "GTA", "GTG"],
        "*": ["TAA", "TAG", "TGA"],
    }

    def __init__(self, design_name: str):
        self.constructs: list[TeselagenDesignConstruct] = []
        self.design_name = design_name

    # ------------------------------------------------------------------
    # public API – build a mutant construct
    # ------------------------------------------------------------------

    def build_mutant(self, starting_genbank_fpath: str, allele_id: str) -> TeselagenDesignConstruct:
        """Create a mutant construct and store it internally."""
        record = SeqIO.read(starting_genbank_fpath, "genbank")
        cds_feature = next(f for f in record.features if f.type == "CDS")
        cds_nt = cds_feature.extract(record.seq)
        cds_aa = cds_nt.translate()

        m = re.match(r"([A-Z])(\d+)([A-Z])", allele_id)
        if not m:
            raise ValueError("Allele ID must look like A63Y")
        wt_ltr, pos_str, mut_ltr = m.groups()
        pos = int(pos_str)

        if cds_aa[pos - 1] != wt_ltr:
            raise ValueError(
                f"Expected {wt_ltr} at AA pos {pos} in {starting_genbank_fpath}, found {cds_aa[pos-1]}"
            )

        current_codon = str(cds_nt[(pos - 1) * 3 : pos * 3])
        new_codon = self._choose_codon(current_codon, mut_ltr, cds_nt, cds_aa)

        # ------- build part list: upstream / mutant / downstream ----------
        cds_start = int(cds_feature.location.start)
        codon_start = cds_start + (pos - 1) * 3
        codon_end = codon_start + 3

        construct_name = f"{Path(starting_genbank_fpath).stem}_{allele_id}"
        construct = TeselagenDesignConstruct(construct_name)

        if codon_start > 0:
            construct.add_part(
                TeselagenDesignPart(gb_tuple=(starting_genbank_fpath, 0, codon_start))
            )
        construct.add_part(TeselagenDesignPart(nucleic_acid_seq=new_codon))
        if codon_end < len(record):
            construct.add_part(
                TeselagenDesignPart(
                    gb_tuple=(starting_genbank_fpath, codon_end, len(record))
                )
            )

        self.constructs.append(construct)
        return construct

    # ------------------------------------------------------------------
    # helper
    # ------------------------------------------------------------------

    def _choose_codon(self, current_codon: str, aa_letter: str, cds_nt: Seq, cds_aa: Seq) -> str:
        aa_codon_counts = {}
        for aa_idx, aa in enumerate(cds_aa):
            if aa == aa_letter:
                codon = cds_nt[aa_idx * 3: (aa_idx + 1) * 3]
                aa_codon_counts[codon] = aa_codon_counts.get(codon, 0) + 1
        if len(aa_codon_counts) == 0:
            raise ValueError(f"No codons found for {aa_letter} in existing gene ({cds_aa})")
        
        codon_counts_sorted = sorted(aa_codon_counts.items(), key=lambda x: x[1], reverse=True)
        most_common_codon = codon_counts_sorted[0][0]
        if most_common_codon.translate() != aa_letter:
            raise ValueError(f"Bug! Most common codon for {aa_letter} in existing gene ({cds_aa}) is {most_common_codon}, which translates to {most_common_codon.translate()}")
        return str(most_common_codon)

    # ------------------------------------------------------------------
    # Teselagen JSON export
    # ------------------------------------------------------------------

    # ------------------------------------------------------------------
    # Export to Teselagen JSON
    # ------------------------------------------------------------------
    def to_teselagen_json(self, assembly_method="golden gate", allow_duplicates=True):
        """Produce Teselagen‑compatible JSON including *all* annotated GenBank parts."""

        part_key_to_id: dict[tuple, str] = {}
        gb_subsets: dict[str, set[tuple[int, int]]] = {}
        nucleic_parts: dict[str, str] = {}

        def _sha_id(txt: str) -> str:
            # return "p_" + hashlib.sha1(txt.encode()).hexdigest()[:8]
            return f'p_{txt}'
        # ---------------- aggregate parts from constructs ----------------
        for cons in self.constructs:
            for part in cons.parts:
                key = part._key
                if key in part_key_to_id:
                    continue
                if key[0] == "gb":
                    _, pth, s, e = key
                    pid = f"{Path(pth).stem}-{s}-{e}"
                    part_key_to_id[key] = pid
                    gb_subsets.setdefault(pth, set()).add((s, e))
                else:  # synthetic sequence
                    _, seq = key
                    pid = _sha_id(seq)
                    part_key_to_id[key] = pid
                    nucleic_parts[pid] = seq

        # ---------------- sequences array ----------------
        sequences_json = []
        for gb_path, subset_set in gb_subsets.items():
            rec = SeqIO.read(gb_path, "genbank")
            seq_name = Path(gb_path).stem
            parts_json = []

            # 2a) add *all* annotated features as parts
            for feat in rec.features:
                if feat.type == "source":
                    continue
                start = int(feat.location.start)
                end = int(feat.location.end) - 1  # Inclusive for Teselagen
                fid = f"{seq_name}_{start}_{end}_{feat.type}"
                fname = feat.qualifiers.get("label", [feat.type])[0]
                parts_json.append({
                    "start": start,
                    "end": end,
                    "id": fid,
                    "name": fname,
                    "strand": 1 if feat.location.strand != -1 else -1,
                })

            # 2b) add subset slices actually referenced by constructs
            for (s, e) in sorted(subset_set):
                pid = part_key_to_id[("gb", Path(gb_path).resolve().as_posix(), s, e)]
                parts_json.append({
                    "start": s,
                    "end": e - 1,
                    "id": pid,
                    "name": pid,
                    "strand": 1,
                })

            sequences_json.append({
                "name": seq_name,
                "sequence": str(rec.seq),
                "parts": parts_json,
                "circular": True,
            })

        # synthetic sequences
        for pid, seq in nucleic_parts.items():
            sequences_json.append({
                "name": pid,
                "sequence": seq,
                "parts": [{"start": 0, "end": len(seq) - 1, "id": pid, "name": pid, "strand": 1}],
            })


        # --------------------------------------------------
        # 3) Build columns (one per construct)
        # --------------------------------------------------
        columns_json = []
        for column_idx in range(max(len(construct.parts) for construct in self.constructs)):
            col_parts = []
            for construct in self.constructs:
                if column_idx < len(construct.parts):
                    part = construct.parts[column_idx]
                    pid = part_key_to_id[part._key]
                    col_parts.append({"id": pid})
                else:
                    col_parts.append({"id": ""})
            columns_json.append({
                "direction": "forward",
                "icon": "cds",
                "name": f'Part {column_idx + 1}',
                "parts": col_parts,
            })

        # --------------------------------------------------
        # 4) Final JSON structure
        # --------------------------------------------------
        design_json = {
            "assembly_method": assembly_method,
            "columns": columns_json,
            "layout_type": "list",
            "name": self.design_name,
            "sequences": sequences_json,
            # "lab": "25e3cb66-456b-4abd-b52b-52546206955d"
        }

        return {
            "allowDuplicates": allow_duplicates,
            "designJson": design_json,
        }



In [ ]:
# Populate a builder class
builder = TeselagenDesignBuilder(DESIGN_ID)
for row_idx, row in round2_seq_df.iterrows():
    seq_id = row['seq_id']
    base_seq_id = row['base_seq_id']
    new_allele = row['new_allele_id']

    genbank_fpath = Path(TEMPLATE_PLASMID_REPO) / f'{CAMPAIGN_ID}-{base_seq_id}.gb'
    builder.build_mutant(str(genbank_fpath), new_allele)
builder.to_teselagen_json()

/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '14883..480' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(


{'allowDuplicates': True,
 'designJson': {'assembly_method': 'golden gate',
  'columns': [{'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 1',
    'parts': [{'id': 'TY_Pop2-WT-0-9023'},
     {'id': 'TY_Pop2-WT-0-9359'},
     {'id': 'TY_Pop2-WT-0-9371'},
     {'id': 'TY_Pop2-WT-0-9413'},
     {'id': 'TY_Pop2-WT-0-9515'},
     {'id': 'TY_Pop2-WT-0-9743'},
     {'id': 'TY_Pop2-WT-0-9893'},
     {'id': 'TY_Pop2-WT-0-10073'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8840'},
     {'id': 'TY_Pop2-F61L-0-9608'},
     {'id': 'TY_Pop2-G429R-0-8996'},
     {'id': 'TY_Pop2-G429R-0-9125'},
     {'id': 'TY_Pop2-G429R-0-9125'},
     {'id': 'TY_Pop2-G429R-0-9188'},
     {'id': 'TY_Pop2-G429R-0-9197'},
     {'id': 'TY_Pop2-G429R-0-9215'},
     {'id': 'TY_Pop2-G429R-0-9269'},
     {'id': 'TY_Pop2-G429R-0-9365'},
     {'id': 'TY_Pop2-G429R-0-9515'},
     {'id': 'TY_Pop2-G429R-0-9524'},
     {'i

In [ ]:
import requests
design = builder.to_teselagen_json()

# This script is used to post a design to the Teselagen API.
def post_design(session, design):
    """Fetch notebook entry by id."""
    url = f"{BASE_URL}/designs"
    response = session.post(url, json=design)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []


BASE_URL = "https://jbei.teselagen.com/tg-api"

session: requests.Session = requests.Session()
session.headers.update(
    {"Content-Type": "application/json", "Accept": "application/json"}
)

# Authenticate and get the token
response: requests.Response = session.put(
    url=f"{BASE_URL}/public/auth",
    json={
        "username": USERNAME,
        "password": TESELAGEN_OTP,
        "expiresIn": "1d",
    },
)
response.raise_for_status()  # Raise an error if a problem is found
session.headers.update(
    {
        "x-tg-api-token": response.json()["token"],  # TOKEN
        "tg-project-id": TESELAGEN_PROJECT_ID
    },  
)
session.headers.pop("Content-Type", None)
del response

# post the design
post_design(session, design)


/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '14883..480' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(


{'id': '9d433d73-3cbe-4327-9ec9-de49e2764c48'}